# ExaHyPE2 Tutorials, part 2: the shallow-water equations


We import the required modules for this project.
We always need the 'peano4' module as this is our core project.
Since we are creating an ExaHyPE 2 application, we additionally need to import the 'exahype2' module.

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../../python/"))

import peano4, exahype2

Here, we define an ExaHyPE 2 solver, specifically an ADER-DG solver.
It takes as arguments a name, a polynomial order, the unknowns, and the allowable cell sizes for the refinement.

In [ ]:
unknowns = {"h": 1, "hu": 2, "b": 1}
auxiliary_variables = {}
max_h = 1.1 * 2.0 / (3.0 ** 3)
min_h = max_h

my_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                  = "SWESolver",
    order                 = 5,
    min_cell_h            = min_h,
    max_cell_h            = max_h,
    time_step_relaxation  = 0.5,
    unknowns              = unknowns,
    auxiliary_variables   = auxiliary_variables
)

my_solver.plot_description = ", ".join(unknowns.keys())

With our solver defined, we can let it know what our equation will look like.
`exahype2.solvers.PDETerms.User_Defined_Implementation` lets the project know that we will manually define the terms in an implementation file.

In [ ]:
my_solver.set_implementation(
    flux = exahype2.solvers.PDETerms.User_Defined_Implementation,
    ncp  = exahype2.solvers.PDETerms.User_Defined_Implementation,
    refinement_criterion = exahype2.solvers.PDETerms.User_Defined_Implementation
)

Here, we define an ExaHyPE 2 project to which our solver will be bound.
This takes a namespace for the C++ code and a name for the executable. We then add our solver to this project and, just for cleanliness, set an output path for our results.

In [ ]:
exahype2_project = exahype2.Project(
    namespace     = ["exahype2", "training", "swe"],
    directory     = ".",
    project_name  = "DamBreak",
    executable    = "DamBreak"
)

exahype2_project.add_solver(my_solver)
exahype2_project.set_output_path("solutions")

Now that we have an ExaHyPE 2 project, let's populate it with some information:
This is essentially the domain our solver will run in.
The project needs to know the domain size, how long the simulation should run,
how often it should plot the solution, and what type of boundary we use.

In [ ]:
exahype2_project.set_global_simulation_parameters(
    dimensions            = 2,
    size                  = [ 2.0,  2.0],
    offset                = [-1.0, -1.0],
    min_end_time          = 0.5,
    max_end_time          = 0.5,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.1,
    periodic_BC           = [False, False],
)

In [ ]:
exahype2_project.set_load_balancing(
    "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
    "new ::exahype2::LoadBalancingConfiguration()"
)

exahype2_project.set_Peano4_installation("../../../", peano4.output.CompileMode.Release)
peano4_project = exahype2_project.generate_Peano4_project()
peano4_project.output.makefile.add_CXX_flag("-DExercise=1")
peano4_project.generate()

And finally, we can compile the project.

In this example the implementation is not provided for you, so you will need to look into the implementation file **SWESolver.cpp**, implement the functions and re-compile the project before you can run it. Go do so now!

In [ ]:
peano4_project.build(make=True, make_clean_first=False, throw_away_data_after_build=False)

Once you've implemented the necessary functions and re-compile the project, you can run it.

This is usually done directly in the terminal via `./$executable` but can be done in python as well.

In [ ]:
import subprocess

subprocess.run("./DamBreak", shell = True, executable="/bin/bash")

Finally, once the program has finished running, we can convert the output from the native peano4 format into something that can be visualized.

In [ ]:
import subprocess
subprocess.call(
    ["python3", "../../../python/peano4/visualisation/render.py", "solutions/solution-SWESolver.peano-patch-file"]
)